# 1. Frozen DistilBERT: title, subtitle and full body

This notebook embeds **title, subtitle and full body** with `sentence-transformers/distiluse-base-multilingual-cased-v2`, then retrieves the top 200 by cosine similarity to the normalized mean of the latest 10 past-click vectors. There is no transformer fine-tuning or learned recommendation scorer. The other two named BERT notebooks use the same settings so the text inputs can be compared on identical requests and catalogue rules.

The Parquet submission has one `behaviour_id` per row and an ordered native list of article IDs, highest-ranked first. Scores are saved in a separate Parquet companion. This notebook has its own experiment/output/cache folder. Match EMBEDDING_EXPERIMENT_ID to a previous cache if reusing embeddings generated under another experiment name.


# 2. Setup and run configuration

In Colab, enable a GPU if desired, mount Drive and upload the **new similarity-only `catalogue_pipeline.py`**. The upload option below accepts Colab's renamed filenames and explicitly imports the uploaded file, avoiding stale module copies. `DRIVE_ROOT` must point to the actual shared-folder shortcut in your account.

Start in audit mode. Switch to run mode after checking the paths and costs. Every configuration-cell execution creates a fresh run ID. To switch audit/run without changing it, set RUN_PHASE in a separate cell. Completed runs can be reused without scoring again if their configuration is unchanged.


In [1]:
from pathlib import Path
from datetime import datetime
from zoneinfo import ZoneInfo
import hashlib
import importlib.util
import json
import subprocess
import sys

try:
    IN_COLAB = importlib.util.find_spec('google.colab') is not None
except ModuleNotFoundError:
    IN_COLAB = False

EXPERIMENT_ID = '3.2_pretrained_similarity_bert_full'
if IN_COLAB:
    from google.colab import drive, files
    drive.mount('/content/drive')
    FORCE_UPLOAD = True  # Prompt again when updating the helper.
    CODE_DIR = Path('/content/Experiments') / EXPERIMENT_ID
    CODE_DIR.mkdir(parents=True, exist_ok=True)
    if FORCE_UPLOAD or not (CODE_DIR / 'catalogue_pipeline.py').exists():
        uploaded = files.upload()
        matches = [name for name in uploaded
                   if Path(name).stem.startswith('catalogue_pipeline')
                   and Path(name).suffix == '.py']
        if len(matches) != 1:
            raise ValueError(f'Select exactly one catalogue_pipeline Python file; received {list(uploaded)}')
        (CODE_DIR / 'catalogue_pipeline.py').write_bytes(uploaded[matches[0]])
else:
    candidates = [Path.cwd(), Path.cwd() / 'Experiments' / EXPERIMENT_ID,
                  Path.cwd() / 'Experiments' / '3.2_pretrained_similarity_bert']
    CODE_DIR = next((p for p in candidates if (p / 'catalogue_pipeline.py').exists()), None)
    if CODE_DIR is None:
        raise FileNotFoundError('Open from the repository root or this experiment folder.')

subprocess.run([sys.executable, '-m', 'pip', 'install',
                'numpy>=1.26', 'pandas>=2.2', 'pyarrow>=15',
                'scikit-learn>=1.4', 'sentence-transformers>=3', 'torch>=2.2', 'huggingface-hub>=0.23'], check=True)
import numpy as np
import pandas as pd

helper_path = CODE_DIR / 'catalogue_pipeline.py'
spec = importlib.util.spec_from_file_location('catalogue_pipeline', helper_path)
cp = importlib.util.module_from_spec(spec)
sys.modules['catalogue_pipeline'] = cp
spec.loader.exec_module(cp)
if getattr(cp, 'PIPELINE_VERSION', None) != 'similarity_individual_history_v6':
    raise RuntimeError('Upload the new similarity-only helper, not the archived hybrid version.')
print('Loaded:', cp.__file__)


Mounted at /content/drive


Saving catalogue_pipeline.py to catalogue_pipeline.py
Loaded: /content/Experiments/3.2_pretrained_similarity_bert_full/catalogue_pipeline.py


In [2]:
MODEL_NAME = 'sentence-transformers/distiluse-base-multilingual-cased-v2'
TEXT_VARIANT = 'title_subtitle_body'
EMBEDDING_EXPERIMENT_ID = EXPERIMENT_ID  # Set to your OLD experiment ID only if reusing its matching cache.
EVALUATE_TEST = False  # Enable only after fixing all model settings using validation.
INTRO_CHARACTERS = 1000  # Explicit first-body-character definition, not a parsed paragraph.
ENCODING_BATCH_SIZE = 64  # GPU batch: chunks collected across articles.
ARTICLE_BATCH_SIZE = 128  # CPU article buffer and cache commit interval.

OWNER = ''  # Your name for final handoff.
DRIVE_ROOT = Path('/content/drive/MyDrive/Colab Notebooks/BT4222 Shared Final Folder')
SEED = 42
RUN_PHASE = 'run'  # 'audit' or 'run'
LOAD_RUN = None  # Optional completed similarity-only persistent run to reproduce.
RUN_ID = (
    datetime.now(ZoneInfo('Asia/Singapore')).strftime('%Y%m%d_%H%M%S') + f'_bert_seed{SEED}'
)
WORK = CODE_DIR / 'outputs' / RUN_ID
PERSISTENT_ROOT = DRIVE_ROOT / 'Experiments'
CONFIG = {
    'experiment_id': EXPERIMENT_ID, 'owner': OWNER, 'run_id': RUN_ID,
    'drive_root': str(DRIVE_ROOT), 'seed': SEED, 'scope': 'development',
    'input_version': None,
    'embedding_source': {'provider': 'Sentence Transformers', 'model': MODEL_NAME, 'revision': None, 'text_variant': TEXT_VARIANT, 'intro_characters': INTRO_CHARACTERS},
    'embedding_column': 'text_embedding', 'embedding_dimension': None,
    'history_limit': 10,
    'history_aggregation': 'mean_profile',  # Same baseline rule across all three notebooks.
    'history_top_k': 3,  # Only used if you later choose topk_history.
    'validation_requests': None,
    'export_requests': {'train': 25, 'validation': None, 'test': None},
    'catalogue_batch_size': 4096,
    'similarity_device': 'auto',  # 'cpu', 'cuda', or automatic CUDA/CPU fallback.
    'similarity_min_gpu_batch': 256,
    'hash_inputs': False,
    'max_index_gb': 20, 'max_export_pairs': 1_200_000_000,
    'max_export_gb': 10, 'max_scoring_seconds': 3600,
    'persistent_root': str(PERSISTENT_ROOT),
    'paths': {},  # Override actual Drive input paths here if needed.
}
CONFIG['code_provenance'] = {
    'helper_sha256': hashlib.sha256(helper_path.read_bytes()).hexdigest(),
    'commit': None, 'uncommitted_changes': None,
}
try:
    CONFIG['code_provenance'].update(
        commit=subprocess.run(['git', '-C', str(CODE_DIR), 'rev-parse', 'HEAD'], capture_output=True, text=True, check=True).stdout.strip(),
        uncommitted_changes=bool(subprocess.run(['git', '-C', str(CODE_DIR), 'status', '--porcelain'], capture_output=True, text=True, check=True).stdout),
    )
except (subprocess.CalledProcessError, FileNotFoundError):
    pass
print('Drive root exists:', DRIVE_ROOT.exists())
print('Local run:', WORK)
print('No classifier training or checkpoint is required.')


Drive root exists: True
Local run: /content/Experiments/3.2_pretrained_similarity_bert_full/outputs/20261105_195308_bert_seed42
No classifier training or checkpoint is required.


## Generate or reuse DistilBERT article embeddings

Run this before `cp.prepare`. Choose a GPU runtime for embedding generation. This encodes all articles in both shared catalogues, including historical articles present there. Missing articles outside the shared catalogue remain subject to the existing coverage audit. No click labels are used to encode article text.

The first run downloads the encoder and may take substantially longer than retrieval. Embeddings are saved to your experiment folder on Drive. Repeating the same configuration reuses them; interrupted generation resumes from the last committed article buffer. Change only the encoder first, keeping fields/history/requests the same as Gemini. Confirm Gemini actually used the same fields with the preprocessing owner.

`ARTICLE_BATCH_SIZE = 128` collects articles on the CPU. `ENCODING_BATCH_SIZE = 64` sends up to 64 chunks from those articles to the GPU together. Increase the latter to 128 after checking speed and memory; reduce it if CUDA reports out of memory. Changing batch sizes keeps the same chunk boundaries, text and pooling, so existing cached vectors can be reused (small numerical differences are possible). Only the last uncommitted article buffer needs repeating after an interruption.

Each named notebook uses its own EXPERIMENT_ID by default. If a matching cache already exists under a legacy experiment folder, set EMBEDDING_EXPERIMENT_ID to that folder ID. Encoder, revision, text variant and input fingerprints still need to match. Changing this setting does not convert full-body vectors into title-only vectors.


In [3]:
import sqlite3
from contextlib import closing
import pyarrow as pa
import pyarrow.parquet as pq
import torch
from sentence_transformers import SentenceTransformer
from huggingface_hub import HfApi


def text_for_article(row):
    variants = {
        'title_subtitle_body': ('title', 'subtitle', 'body'),
        'title_subtitle': ('title', 'subtitle'),
        'title_subtitle_intro': ('title', 'subtitle', 'body'),
    }
    if TEXT_VARIANT not in variants:
        raise ValueError(f'Unknown text variant: {TEXT_VARIANT}')
    fields = []
    for name in variants[TEXT_VARIANT]:
        value = row.get(name)
        if isinstance(value, str) and value.strip():
            value = value.strip()
            if name == 'body' and TEXT_VARIANT == 'title_subtitle_intro':
                value = value[:INTRO_CHARACTERS]
            fields.append(f'{name}: {value}')
    return '\n'.join(fields) or None


def encode_articles(texts, encoder):
    """Batch chunks across articles, preserving a separate weighted mean per article."""
    if ENCODING_BATCH_SIZE < 1:
        raise ValueError('ENCODING_BATCH_SIZE must be positive')
    tokenizer = encoder.tokenizer
    totals = np.zeros((len(texts), encoder.get_sentence_embedding_dimension()), dtype='float64')
    pending = []

    def flush():
        if not pending:
            return
        # Dynamic padding is only for this GPU batch, not the whole article buffer.
        features = {}
        for name in tokenizer.model_input_names:
            if name in pending[0][2]:
                padding = (getattr(tokenizer, 'pad_token_id', 0) or 0) if name == 'input_ids' else 0
                features[name] = torch.nn.utils.rnn.pad_sequence(
                    [row[2][name] for row in pending], batch_first=True,
                    padding_value=padding).to(encoder.device)
        with torch.inference_mode():
            output = encoder(features)['sentence_embedding']
            vectors = torch.nn.functional.normalize(output.float(), p=2, dim=1).cpu().numpy()
        for (article_index, weight, _), vector in zip(pending, vectors):
            totals[article_index] += vector.astype('float64') * weight
        pending.clear()

    for article_index, text in enumerate(texts):
        if not text:
            continue
        chunks = tokenizer(text, truncation=True, max_length=encoder.max_seq_length,
                           stride=0, return_overflowing_tokens=True, padding=True,
                           return_special_tokens_mask=True, return_tensors='pt')
        weights = (chunks['attention_mask'] * (1 - chunks['special_tokens_mask'])).sum(1)
        for chunk_index, weight in enumerate(weights.tolist()):
            if weight <= 0:
                continue
            length = int(chunks['attention_mask'][chunk_index].sum())
            features = {name: chunks[name][chunk_index, :length]
                        for name in tokenizer.model_input_names if name in chunks}
            pending.append((article_index, weight, features))
            if len(pending) >= ENCODING_BATCH_SIZE:
                flush()
    flush()
    result = []
    for text, total in zip(texts, totals):
        if not text:
            result.append(None)
            continue
        norm = np.linalg.norm(total)
        if not np.isfinite(total).all() or norm <= 1e-12:
            raise ValueError('Encoder produced an unusable article vector')
        result.append((total / norm).astype('float32'))
    return result


# Resolve the exact revision once. Existing cache metadata pins later runs to it.
CACHE_ROOT = DRIVE_ROOT / 'Experiments' / EMBEDDING_EXPERIMENT_ID / 'embeddings'
CACHE_ROOT.mkdir(parents=True, exist_ok=True)
CACHE = CACHE_ROOT / (MODEL_NAME.split('/')[-1] + '_' + TEXT_VARIANT
                     + (f'_{INTRO_CHARACTERS}chars' if TEXT_VARIANT.endswith('intro') else ''))
CACHE.mkdir(exist_ok=True)
metadata_path = CACHE / 'embedding_config.json'
sources = {part: DRIVE_ROOT / part / 'articles.parquet' for part in ('train_val', 'test')}
for path in sources.values():
    if not path.is_file():
        raise FileNotFoundError(f'Check DRIVE_ROOT: {path}')
source_hashes = {part: cp.fingerprint(path, content_hash=True) for part, path in sources.items()}
old_metadata = json.loads(metadata_path.read_text()) if metadata_path.exists() else None
MODEL_REVISION = (old_metadata['model_revision'] if old_metadata
                  else HfApi().model_info(MODEL_NAME).sha)
embedding_config = {
    'model': MODEL_NAME, 'model_revision': MODEL_REVISION,
    'text_variant': TEXT_VARIANT, 'intro_characters': INTRO_CHARACTERS,
    'sources': source_hashes, 'snapshot_policy': 'train_val first, test second',
    'pooling': 'nonoverlapping token chunks; normalized chunk vectors; token-weighted mean; normalize',
}
if old_metadata and any(old_metadata.get(k) != v for k, v in embedding_config.items()):
    raise ValueError('Embedding inputs/config changed. Use a new CACHE directory to preserve prior vectors.')

EMBEDDING_PATHS = {part: CACHE / f'{part}_text_embeddings.parquet' for part in sources}
complete = old_metadata and old_metadata.get('complete') and all(p.is_file() for p in EMBEDDING_PATHS.values())
if complete:
    print('Reusing saved embeddings:', CACHE)
    dimension = old_metadata['dimension']
else:
    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    print('Generating embeddings on:', device, '| This processes the full article catalogues, not just 1,000 requests.')
    encoder = SentenceTransformer(MODEL_NAME, revision=MODEL_REVISION, device=device)
    encoder.eval()
    dimension = encoder.get_sentence_embedding_dimension()
    embedding_config.update(dimension=dimension, max_sequence_length=encoder.max_seq_length, complete=False,
                            encoding_batch_size=ENCODING_BATCH_SIZE, article_batch_size=ARTICLE_BATCH_SIZE,
                            batching='chunks pooled across articles')
    cp.save_json(metadata_path, embedding_config)
    # Commit each article buffer as one transaction; interrupted work resumes at the last commit.
    with closing(sqlite3.connect(CACHE / 'article_vectors.sqlite')) as db:
        db.execute('CREATE TABLE IF NOT EXISTS vectors (article_id INTEGER PRIMARY KEY, text_hash TEXT, vector BLOB)')
        processed = 0
        for part, source in sources.items():
            parquet = pq.ParquetFile(source)
            required = ['article_id', 'title', 'subtitle']
            if TEXT_VARIANT != 'title_subtitle': required.append('body')
            missing = set(required) - set(parquet.schema_arrow.names)
            if missing: raise ValueError(f'{source}: missing text columns {missing}')
            if ARTICLE_BATCH_SIZE < 1:
                raise ValueError('ARTICLE_BATCH_SIZE must be positive')
            for batch in parquet.iter_batches(batch_size=ARTICLE_BATCH_SIZE, columns=required):
                # Keep the first snapshot, including repeated IDs within a source batch.
                articles = {}
                for row in batch.to_pylist():
                    articles.setdefault(int(row['article_id']), row)
                placeholders = ','.join('?' for _ in articles)
                cached = {row[0] for row in db.execute(
                    f'SELECT article_id FROM vectors WHERE article_id IN ({placeholders})', list(articles))}
                pending_articles = [(aid, text_for_article(row)) for aid, row in articles.items() if aid not in cached]
                if not pending_articles:
                    continue
                vectors = encode_articles([text for _, text in pending_articles], encoder)
                records = [(aid, hashlib.sha256(text.encode()).hexdigest() if text else None,
                            vector.tobytes() if vector is not None else None)
                           for (aid, text), vector in zip(pending_articles, vectors)]
                with db:
                    db.executemany('INSERT INTO vectors VALUES (?,?,?)', records)
                previous = processed
                processed += len(records)
                if processed // 500 > previous // 500:
                    print('New articles embedded:', processed, flush=True)
        schema = pa.schema([('article_id', pa.int64()), ('text_embedding', pa.list_(pa.float32())),
                            ('has_text_embedding', pa.bool_())])
        for part, source in sources.items():
            target = EMBEDDING_PATHS[part]
            temporary = target.with_suffix('.tmp.parquet')
            seen = set()
            with pq.ParquetWriter(temporary, schema) as writer:
                for batch in pq.ParquetFile(source).iter_batches(batch_size=512, columns=['article_id']):
                    rows = []
                    for aid in batch.column(0).to_pylist():
                        aid = int(aid)
                        if aid in seen: raise ValueError(f'Duplicate article ID in {source}: {aid}')
                        seen.add(aid)
                        value = db.execute('SELECT vector FROM vectors WHERE article_id=?', (aid,)).fetchone()
                        if value is None: raise ValueError(f'Missing cached article {aid}')
                        vector = np.frombuffer(value[0], dtype='float32').tolist() if value[0] is not None else None
                        rows.append({'article_id': aid, 'text_embedding': vector, 'has_text_embedding': vector is not None})
                    writer.write_table(pa.Table.from_pylist(rows, schema=schema))
            temporary.replace(target)
    embedding_config['complete'] = True
    cp.save_json(metadata_path, embedding_config)
    del encoder
    if torch.cuda.is_available(): torch.cuda.empty_cache()

CONFIG['embedding_dimension'] = dimension
CONFIG['embedding_source'].update(revision=MODEL_REVISION,
                                  pooling=embedding_config['pooling'],
                                  metadata_path=str(metadata_path))
for part, path in EMBEDDING_PATHS.items():
    CONFIG['paths'][f'embeddings_{part}'] = str(path)
print('Embedding paths:', CONFIG['paths'])
print('Continue with shared-data preparation below.')


Reusing saved embeddings: /content/drive/MyDrive/Colab Notebooks/BT4222 Shared Final Folder/Experiments/3.2_pretrained_similarity_bert_full/embeddings/distiluse-base-multilingual-cased-v2_title_subtitle_body
Embedding paths: {'embeddings_train_val': '/content/drive/MyDrive/Colab Notebooks/BT4222 Shared Final Folder/Experiments/3.2_pretrained_similarity_bert_full/embeddings/distiluse-base-multilingual-cased-v2_title_subtitle_body/train_val_text_embeddings.parquet', 'embeddings_test': '/content/drive/MyDrive/Colab Notebooks/BT4222 Shared Final Folder/Experiments/3.2_pretrained_similarity_bert_full/embeddings/distiluse-base-multilingual-cased-v2_title_subtitle_body/test_text_embeddings.parquet'}
Continue with shared-data preparation below.


# 3. Load and check shared data

Load the shared text embedding lookups, article IDs/publication times, fixed click histories, and original request/candidate keys. The engineered feature tables supply **only IDs, timestamps and genuine click labels for evaluation**. Feature vocabulary, metadata predictors and scaled/log variants are not required.

Article catalogue IDs and vector IDs are audited separately. Duplicate source keys, unreadable vectors, and temporal history violations raise errors. Test labels are ignored during inference and read separately only by the held-out test evaluation section. Original test impression IDs can repeat across users; the request key is `user_id:impression_id`.

Eligibility is unchanged: use the declared ordered catalogue union, retain the first snapshot per article ID, and require known publication time no later than the request. There is no extra age or access cutoff. Missing text vectors remain eligible with a documented zero score. Confirm the same catalogue/policy across the group before model comparison.


In [4]:
PREPARED = cp.prepare(CONFIG, WORK)
audit = PREPARED['audit']
print('Pipeline:', audit['pipeline_version'])
print('Compute:', audit['compute'])
print('Catalogue articles:', audit['catalogue_articles'])
print('Catalogue articles missing vectors:', audit['catalogue_articles_without_vectors'])
print('Embedding dimension:', audit['embedding_info']['dimension'])
display(pd.DataFrame(audit['request_counts']).T)


Similarity-only compute: {'requested_device': 'auto', 'similarity_device': 'cuda', 'classifier_device': 'not_applicable', 'loading_and_exports_device': 'cpu', 'minimum_gpu_batch': 256, 'reason': 'Shared CUDA article table ready', 'article_table_bytes': 42436608, 'torch_version': '2.11.0+cu130', 'cuda_version': '13.0', 'gpu_name': 'Tesla T4'}
Pipeline: similarity_individual_history_v6
Compute: {'requested_device': 'auto', 'similarity_device': 'cuda', 'classifier_device': 'not_applicable', 'loading_and_exports_device': 'cpu', 'minimum_gpu_batch': 256, 'reason': 'Shared CUDA article table ready', 'article_table_bytes': 42436608, 'torch_version': '2.11.0+cu130', 'cuda_version': '13.0', 'gpu_name': 'Tesla T4'}
Catalogue articles: 20738
Catalogue articles missing vectors: 18
Embedding dimension: 512


,candidate_rows,requests,split_start,source_has_clicked_column,outcome_policy,outcome_column_loaded,sampling_rule
train,2585747,232887,2023-05-18T07:00:01,True,observed_labels_for_evaluation,True,Smallest BLAKE2b-64(seed:split:behaviour_id); ...
validation,344666,29110,2023-05-25T07:00:15,True,observed_labels_for_evaluation,True,Smallest BLAKE2b-64(seed:split:behaviour_id); ...
test,348951,29142,2023-05-25T07:00:10,True,ignored_for_test_inference,False,Smallest BLAKE2b-64(seed:split:behaviour_id); ...


In [5]:
print('Planned development exports:')
display(pd.DataFrame(audit['planned_exports']).T[['requests', 'eligible_pairs', 'estimated_csv_bytes', 'empty_pools']])
print('Full-split costs (not the default export scope):')
display(pd.DataFrame(audit['full_export_costs']).T[['requests', 'eligible_pairs', 'estimated_csv_bytes', 'empty_pools']])
cp.check_cost(PREPARED)


Planned development exports:


,requests,eligible_pairs,estimated_csv_bytes,empty_pools
train,25,463808,282500,0
validation,29110,566218581,328943000,0
test,29142,566598922,329304600,0


Full-split costs (not the default export scope):


,requests,eligible_pairs,estimated_csv_bytes,empty_pools
train,232887,4319178324,2631623100,0
validation,29110,566218581,328943000,0
test,29142,566598922,329304600,0


{'pairs': 1133281311,
 'estimated_csv_bytes': 658530100,
 'local_free_bytes': 120101187584}

# 4. Prepare model inputs

For each user, take their eligible past-click article embeddings. Each article vector is L2-normalised; average the selected vectors and normalise that mean to form the user profile. Repeated clicks retain repeated weight. If `history_limit` is set, select the latest N actual clicks by timestamp before dropping missing vectors.

History is a fixed snapshot strictly before the first shared request in each split. It is not updated with current behavior labels. Missing history, no available vectors, or a cancelled mean gives no profile. Such requests receive zero scores with deterministic article-ID ties, and are counted explicitly.

Cosine similarity alone ranks candidates. The availability/coverage flags are diagnostics, not inputs to a learned model. Candidate metadata, current/subsequent read outcomes and click labels do not change scores.


In [6]:
print('Scoring rule: score(user, article) = normalised_article_vector @ normalised_user_profile')
print('History setting:', CONFIG['history_limit'] or 'all eligible past clicks')
print('Fallback: score 0 for unavailable profile or candidate vector; retain eligible articles.')


Scoring rule: score(user, article) = normalised_article_vector @ normalised_user_profile
History setting: 10
Fallback: score 0 for unavailable profile or candidate vector; retain eligible articles.


# 5. Define and train / load model

**There is nothing to train.** No epochs, alpha trials, classifier checkpoints, scalers or learned feature weights are used. This section saves a small JSON inference specification or loads the same specification from a completed similarity-only run. Old hybrid `selected.joblib` files are incompatible and are not read.

Set `RUN_PHASE='run'` after the audit. Rerun sections 3-4 if changing configuration. Use a fresh run ID for a different encoder, text variant or history setting. No `cp.train()` call is needed.


In [7]:
MODEL = None
if PREPARED['work'] != WORK or PREPARED['config'] != CONFIG:
    raise RuntimeError('Rerun sections 3-4 after changing configuration.')
if RUN_PHASE == 'run':
    MODEL = cp.load_run(PREPARED, LOAD_RUN) if LOAD_RUN else cp.define_model(PREPARED)
    print(json.dumps(MODEL, indent=2))
else:
    print('Audit only. Set RUN_PHASE="run" after reviewing costs.')


{
  "pipeline_version": "similarity_individual_history_v6",
  "method": "mean_profile_cosine",
  "history_aggregation": "mean_profile",
  "history_top_k": null,
  "history_limit": 10,
  "article_normalization": "L2 per nonmissing article vector",
  "user_profile": "L2-normalized mean of selected past-click article vectors",
  "history_snapshot": "strictly before first request of each split; no behavior-log updates",
  "repeated_clicks": "retain repeated weight/events, including in top-k aggregation",
  "missing_vectors_or_history": "score 0; retain eligible articles; report flags and ties",
  "score_semantics": "aggregated cosine similarity in [-1,1], not a probability",
  "learned_parameters": false,
  "fine_tuned_encoder": false,
  "shortlist_size": 200,
  "tie_policy": "descending cosine, ascending original article ID"
}


# 6. Validation diagnostics and explainability

Evaluate the fixed validation-request sample against its **full eligible catalogue**. Primary metric: mean known-click Recall@200. Secondary: observed-click nDCG@200. Keep excluded original known clicks in both denominators/ideals. Empty pools with known clicks receive zero. Unshown outcomes remain unknown, not negative training labels.

There is no automatic tuning search in this notebook. If you later compare history settings, encoders or text variants, fix the same request IDs and select using validation Recall@200, then nDCG@200. Never select using test outcomes. Previously inspected/repeated validation is development evidence, not an untouched final test.


In [8]:
VALIDATION = None
if RUN_PHASE == 'run':
    VALIDATION = cp.evaluate(PREPARED, MODEL)
    display(pd.json_normalize(VALIDATION['metrics']))
    print('Scoring seconds:', round(VALIDATION['seconds'], 2))
    print('Catalogue pairs scored:', VALIDATION['eligible_pairs'])
    print('History coverage:', VALIDATION['history_coverage'])


,known_click_recall@200,observed_click_ndcg@200,evaluated_requests,requests_without_known_clicks,empty_eligible_pools,known_clicks_excluded_by_eligibility,known_clicks,eligible_only_recall@200,requests_with_ties_crossing_200,known_click_eligibility_coverage,ties,outcomes,requests_without_usable_user_profile,known_click_exclusion_reasons.missing_article,known_click_exclusion_reasons.missing_publication,known_click_exclusion_reasons.future_publication
0,0.017159,0.003097,29110,0,0,0,29249,0.017159,3,1.0,"descending score, ascending original article I...",unshown outcomes unknown; zero observed gain i...,0,0,0,0


Scoring seconds: 583.22
Catalogue pairs scored: 566218581
History coverage: {'users_requested': 1719, 'users_found': 1719, 'missing_history_users': 0, 'events_missing_time_excluded': 0, 'selected_history_events': 16906, 'selected_events_with_vectors': 16906, 'history_limit': 10, 'history_aggregation': 'mean_profile', 'history_top_k': None}


Write `rankings/{split}/submission_top_200.parquet` directly with **behaviour_id,article_ids**: one request per row, native integer lists, highest-ranked article first. Scores are stored separately in `submission_top_200_scores.parquet`, aligned with the same IDs. No submission CSV or exhaustive catalogue-score file is created. Existing eligibility CSVs are internal request audits.


In [9]:
RESULT = None
if RUN_PHASE == 'run':
    RESULT = cp.execute(PREPARED, MODEL)
    display(pd.json_normalize(RESULT['metrics']))
    print('Exported-validation metrics:')
    display(pd.json_normalize(RESULT['exported_validation']))
    print('Export checks:', RESULT['checks'])
    print('Complete final handoff:', RESULT['handoff_complete'])


,known_click_recall@200,observed_click_ndcg@200,evaluated_requests,requests_without_known_clicks,empty_eligible_pools,known_clicks_excluded_by_eligibility,known_clicks,eligible_only_recall@200,requests_with_ties_crossing_200,known_click_eligibility_coverage,ties,outcomes,requests_without_usable_user_profile,known_click_exclusion_reasons.missing_article,known_click_exclusion_reasons.missing_publication,known_click_exclusion_reasons.future_publication
0,0.017159,0.003097,29110,0,0,0,29249,0.017159,3,1.0,"descending score, ascending original article I...",unshown outcomes unknown; zero observed gain i...,0,0,0,0


Exported-validation metrics:


,known_click_recall@200,observed_click_ndcg@200,evaluated_requests,requests_without_known_clicks,empty_eligible_pools,known_clicks_excluded_by_eligibility,known_clicks,known_click_eligibility_coverage,eligible_only_recall@200,requests_with_ties_crossing_200,source,known_click_exclusion_reasons.missing_article,known_click_exclusion_reasons.missing_publication,known_click_exclusion_reasons.future_publication
0,0.017159,0.003097,29110,0,0,0,29249,1.0,0.017159,3,validated exported catalogue shortlists; origi...,0,0,0


Export checks: {'train': {'catalogue_pairs_rescored_for_verification': 463808, 'requests_verified': 25, 'shortlisted_articles_verified': 5000, 'full_score_rows_exported': 0}, 'validation': {'catalogue_pairs_rescored_for_verification': 566218581, 'requests_verified': 29110, 'shortlisted_articles_verified': 5822000, 'full_score_rows_exported': 0}, 'test': {'catalogue_pairs_rescored_for_verification': 566598922, 'requests_verified': 29142, 'shortlisted_articles_verified': 5828400, 'full_score_rows_exported': 0}}
Complete final handoff: False


## Parquet submission files

New runs write native Parquet directly. The next cell lists/previews those files and copies them into an existing persistent bundle if needed. It also supports converting legacy CSV outputs without rescoring: keep WORK pointing to that completed run, reload the updated helper and run only this cell. No embedding generation or catalogue scoring is needed for conversion.


In [10]:
completed_result = globals().get('RESULT')
if completed_result is None and (WORK / 'result.json').is_file():
    completed_result = json.loads((WORK / 'result.json').read_text())
persistent = completed_result['persistence']['persistent_location'] if completed_result and completed_result['persistence']['persistent_copy_complete'] else None
PARQUET_SUBMISSIONS = cp.export_parquet_submissions(WORK, persistent)
print(json.dumps(PARQUET_SUBMISSIONS, indent=2))
display(pd.read_parquet(WORK / 'rankings/validation/submission_top_200.parquet').head())


{
  "train": {
    "path": "/content/Experiments/3.2_pretrained_similarity_bert_full/outputs/20261105_195308_bert_seed42/rankings/train/submission_top_200.parquet",
    "scores_path": "/content/Experiments/3.2_pretrained_similarity_bert_full/outputs/20261105_195308_bert_seed42/rankings/train/submission_top_200_scores.parquet",
    "rows": 25,
    "article_ids": 5000,
    "schema": "behaviour_id:string,article_ids:list<int64>"
  },
  "validation": {
    "path": "/content/Experiments/3.2_pretrained_similarity_bert_full/outputs/20261105_195308_bert_seed42/rankings/validation/submission_top_200.parquet",
    "scores_path": "/content/Experiments/3.2_pretrained_similarity_bert_full/outputs/20261105_195308_bert_seed42/rankings/validation/submission_top_200_scores.parquet",
    "rows": 29110,
    "article_ids": 5822000,
    "schema": "behaviour_id:string,article_ids:list<int64>"
  },
  "test": {
    "path": "/content/Experiments/3.2_pretrained_similarity_bert_full/outputs/20261105_195308_bert_

,behaviour_id,article_ids
0,393380:497194197,"[9078646, 9390537, 9625415, 9427204, 9780195, ..."
1,340393:212944368,"[9078646, 9780195, 8902081, 9443589, 7695476, ..."
2,1895052:36672692,"[9078646, 8902081, 9779417, 9337779, 7695476, ..."
3,450639:469050628,"[9078646, 9429576, 7695476, 9778915, 8902081, ..."
4,340393:212944369,"[9078646, 9780195, 8902081, 9443589, 7695476, ..."


## Held-out test evaluation after configuration selection

Run this section only after choosing the encoder, text variant and history setting using validation results. It reads the updated genuine test labels separately and evaluates the already saved top-200 lists. It does not change embeddings, profiles or rankings. Placeholder labels are not valid held-out outcomes.

The current configuration exports **25 test requests**, so the report below covers those 25, not the full test set. For a larger sample change `export_requests["test"]`; use `None` for all test requests. Start a fresh run and review the audit, comparison budget and runtime limit before generating more test recommendations. Setting `test` to `None` can require substantially more work than the current 150-million-comparison budget. Keep the same request IDs/catalogue/policy across group models.

The separate `test_metrics.json` is copied to the persistent run folder. Existing `metrics.json`/`result.json` remain the original validation/inference reports; use the separate test report for held-out results. Unshown articles have unknown outcomes, and no test AUC is computed here.

Test evaluation is disabled by default (`EVALUATE_TEST=False`). Enable it only after selecting text/history settings using validation. All three notebooks retain the same optional test-evaluation code.


In [11]:
TEST_METRICS = None
if RUN_PHASE == 'run' and EVALUATE_TEST:
    import csv
    import pyarrow.parquet as pq

    # Evaluate the completed export. Test labels never enter the ranking index.
    record = json.loads((WORK / 'run_config.json').read_text())
    expected_source = record['inputs']['features_test']
    test_path = Path(expected_source['path'])
    if cp.fingerprint(test_path, content_hash='sha256' in expected_source) != expected_source:
        raise ValueError('Test source changed since export. Generate a fresh run before evaluating.')
    ranking_path = WORK / 'rankings/test/submission_top_200.parquet'
    rankings = {}
    for batch in pq.ParquetFile(ranking_path).iter_batches(batch_size=512):
        for row in batch.to_pylist():
            key = row['behaviour_id']
            ids = row['article_ids']
            if key in rankings or not isinstance(ids, list) or len(ids) > 200:
                raise ValueError('Duplicate request or invalid test shortlist')
            if any(type(aid) is not int for aid in ids) or len(set(ids)) != len(ids):
                raise ValueError('Invalid or duplicate article IDs')
            rankings[key] = ids
    if not rankings:
        raise ValueError('No exported test requests to evaluate')

    positives = {key: set() for key in rankings}
    seen = set()
    columns = ['user_id', 'impression_id', 'candidate_article_id', 'split', 'clicked']
    for batch in pq.ParquetFile(test_path).iter_batches(columns=columns, batch_size=8192):
        for row in batch.to_pylist():
            if row['split'] != 'test':
                raise ValueError('Test source contains another split')
            key = cp.identity(row['user_id'], row['impression_id'])
            if key not in rankings:
                continue
            label = row['clicked']
            if label not in (0, 1):
                raise ValueError('Test evaluation requires genuine, non-null binary click labels')
            seen.add(key)
            if label == 1:
                positives[key].add(int(row['candidate_article_id']))
    if seen != set(rankings):
        raise ValueError('Some exported requests have no matching test labels')

    catalogue = PREPARED['catalogue']
    details = {}
    with (WORK / 'eligibility/test_requests.csv').open(newline='') as stream:
        for row in csv.DictReader(stream):
            key = row['behaviour_id']
            if key in details:
                raise ValueError('Duplicate eligibility request')
            details[key] = row
    if set(details) != set(rankings):
        raise ValueError('Test shortlist and eligibility request coverage differ')

    recalls, ndcgs, eligible_recalls = [], [], []
    reasons = {'missing_article': 0, 'missing_publication': 0, 'future_publication': 0}
    total_known = excluded = without_clicks = empty = ties = 0
    per_request = []
    for key, ranking in rankings.items():
        detail = details[key]
        when = pd.Timestamp(detail['impression_time'])
        eligible = set(cp.eligible_ids(catalogue, when))
        if any(aid not in eligible for aid in ranking):
            raise ValueError('Export contains ineligible article IDs')
        if len(ranking) != min(200, len(eligible)):
            raise ValueError('Export does not contain the expected shortlist size')
        empty += not eligible
        ties += detail['ties_crossing_200'] == 'True'
        known = positives[key]
        if not known:
            without_clicks += 1
            continue
        recall, ndcg = cp.retrieval_metrics(ranking, known)
        recalls.append(recall)
        ndcgs.append(ndcg)
        total_known += len(known)
        for aid in known - eligible:
            if aid not in catalogue.index:
                reasons['missing_article'] += 1
            elif pd.isna(catalogue.loc[aid, 'published_time']):
                reasons['missing_publication'] += 1
            else:
                reasons['future_publication'] += 1
        eligible_known = known & eligible
        excluded += len(known - eligible_known)
        if eligible_known:
            eligible_recalls.append(len(set(ranking) & eligible_known) / len(eligible_known))
        per_request.append({'behaviour_id': key, 'known_clicks': len(known),
                            'recall@200': recall, 'ndcg@200': ndcg})

    TEST_METRICS = {
        'known_click_recall@200': float(np.mean(recalls)) if recalls else None,
        'observed_click_ndcg@200': float(np.mean(ndcgs)) if ndcgs else None,
        'exported_requests': len(rankings), 'evaluated_requests': len(recalls),
        'requests_without_known_clicks': without_clicks, 'known_clicks': total_known,
        'known_clicks_excluded_by_eligibility': excluded,
        'known_click_eligibility_coverage': 1 - excluded / total_known if total_known else None,
        'eligible_only_recall@200': float(np.mean(eligible_recalls)) if eligible_recalls else None,
        'known_click_exclusion_reasons': reasons, 'empty_eligible_pools': empty,
        'requests_with_ties_crossing_200': ties,
        'source': 'post-hoc held-out labels matched to saved test shortlists',
        'outcomes': 'unshown outcomes unknown; original known-click denominators retained',
    }
    report = {'metrics': TEST_METRICS, 'per_request': per_request,
              'source': expected_source, 'run_id': record['config']['run_id'],
              'model': record['model'], 'eligibility_policy': record['eligibility_policy'],
              'request_ids_sha256': hashlib.sha256(json.dumps(sorted(rankings)).encode()).hexdigest(),
              'note': 'Run only after fixing the model using validation. Labels did not affect saved rankings.'}
    cp.save_json(WORK / 'test_metrics.json', report)
    if RESULT and RESULT['persistence']['persistent_copy_complete']:
        destination = Path(RESULT['persistence']['persistent_location'])
        cp.save_json(destination / 'test_metrics.json', report)
        print('Saved held-out test report:', destination / 'test_metrics.json')
    display(pd.json_normalize(TEST_METRICS))
    print('Test requests evaluated:', len(rankings))
    print('This report covers only exported test requests; test=None exports the entire split.')


# 8. Reload check

The full run bundle is copied to shared Drive, with the JSON inference model, code, source versions and a score probe. The helper reloads code/configuration from that persistent copy, reopens the versioned text embedding tables, recomputes probe cosine scores and checks agreement within 2e-6. No classifier is fitted or loaded. Temporary SQLite indexes are rebuilt from the recorded source files when needed rather than being part of the handoff model.

Partial/failed persistence is recorded as incomplete. Missing real-data access cannot be replaced by synthetic performance results. CUDA is optional and its hardware parity check is skipped locally when no CUDA hardware is available.


In [12]:
if RESULT is not None:
    print(json.dumps(RESULT['persistence'], indent=2))
    if RESULT['persistence']['persistent_copy_complete']:
        saved = Path(RESULT['persistence']['persistent_location'])
        print('Saved fixed inference model:', saved / 'model/model_config.json')
        print('No learned model weights/checkpoint are needed.')


{
  "persistent_copy_complete": true,
  "persistent_location": "/content/drive/MyDrive/Colab Notebooks/BT4222 Shared Final Folder/Experiments/3.2_pretrained_similarity_bert_full/20261105_195308_bert_seed42",
  "copied_files_verified_by_size": 19,
  "reload_from_persistent_copy": true,
  "embedding_lookup_reopened": true,
  "learned_parameters": false,
  "rtol": 2e-06,
  "atol": 2e-06,
  "probe_rows": 10
}
Saved fixed inference model: /content/drive/MyDrive/Colab Notebooks/BT4222 Shared Final Folder/Experiments/3.2_pretrained_similarity_bert_full/20261105_195308_bert_seed42/model/model_config.json
No learned model weights/checkpoint are needed.


# 9. Business interpretation and handoff

Report whether direct pretrained content similarity recovers known clicked news in the catalogue top 200, the effect of unavailable profiles/vectors, observed-click ranking metrics, compute and storage cost. The Stage 2 owner receives the top-200 IDs plus the article/user information needed for the separate freshness/diversity/continuity re-ranker.

A top-200 similarity score is not click probability and does not establish retention gains. Do not compare these catalogue results directly with older recorded-slate hybrid results. The shared Drive data has not been run in the local coding environment; implementation verification uses synthetic fixtures only. Run the notebook on the actual shared files before reporting project metrics or a complete persistent handoff.


In [13]:
if RESULT is not None:
    example = json.loads((WORK / 'validation_example.json').read_text())
    display(pd.DataFrame(example['articles']))
    print('Validation request:', example['behaviour_id'])
    print(example['note'])
    print('Bundle:', RESULT['persistence'].get('persistent_location', str(WORK)))
    print('Held-out test report:', WORK / 'test_metrics.json')


,article_id,cosine_similarity,candidate_vector_missing,user_profile_missing,history_vector_coverage
0,9078646,0.778356,False,False,1.0
1,9390537,0.752128,False,False,1.0
2,9625415,0.728832,False,False,1.0
3,9427204,0.728004,False,False,1.0
4,9780195,0.727063,False,False,1.0


Validation request: 393380:497194197
Scores are direct similarity, with no learned weighting or click probability.
Bundle: /content/drive/MyDrive/Colab Notebooks/BT4222 Shared Final Folder/Experiments/3.2_pretrained_similarity_bert_full/20261105_195308_bert_seed42
Held-out test report: /content/Experiments/3.2_pretrained_similarity_bert_full/outputs/20261105_195308_bert_seed42/test_metrics.json
